# AI Workshop - MLCon Berlin 2025

## What this notebook covers

This workshop walks you through building an **Agentic RAG (Retrieval Augmented Generation)** system:

1. **Setup & LLM Clients** - Configure OpenAI and Groq clients
2. **Basic LLM Calls** - Test chat completions with both providers
3. **Document Loading** - Load FAQ documents for our knowledge base
4. **Search Index** - Build a searchable index using minsearch
5. **Tool Definition** - Define a search tool for the LLM to use
6. **Manual RAG** - Build context-augmented prompts manually
7. **Agentic RAG** - Let the LLM decide when to search (tool calling)
8. **Agentic Loop** - Automate the tool-calling cycle

## Next steps (to migrate from reference notebook)
- Add `make_call` helper function
- Add `developer_prompt` for better agent behavior  
- Add automated agentic loop with `while True`
- Explore `toyaikit` for chat interfaces
- OpenAI Agents SDK integration
- Pydantic AI integration
- MCP (Model Context Protocol) server

---

## 1. Setup & Dependencies

In [ ]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

openai_client = OpenAI()

## 2. LLM Clients Setup

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Tell me a short Christmas story"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Groq client (OpenAI-compatible API)
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

# Test Groq client
response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Tell me a short HannukaH story"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Choose which client to use for the rest of the notebook
client = groq_client  # or groq_client

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
response = client.responses.create(
    model="openai/gpt-oss-20b",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()
print(documents_raw)

## 3. Document Loading & Indexing

In [ ]:
documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [ ]:
documents[12]

In [ ]:
from minsearch import AppendableIndex

In [ ]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results

## 4. Search Function & Tool Definition

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
question = 'I just discovered the course. Can I join it now?'

In [ ]:
result = search(question)
print(result)

In [ ]:
prompt = f"""
Answer the question from the student using the provided context

<QUESTION>{question}</QUESTION>

<CONTEXT>{json.dumps(result)}</CONTEXT>
"""

## 5. Manual RAG - Building Context Prompts

In [ ]:
# agentic RAG

chat_messages = [
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

## 6. Agentic RAG - Tool Calling

In [ ]:
print(response)

In [ ]:
tool_call = response.output[0]
tool_call

In [ ]:
chat_messages.append(tool_call)

In [ ]:
search_result = search(query="Can I join the course now?")

In [ ]:
result_json = json.dumps(search_result, indent=2)

chat_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": result_json,
})

In [ ]:
chat_messages

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

In [ ]:
response.output_text

In [ ]:
chat_messages.append(
    {"role": "user", "content": "but are you sure I can get my certificate?"}
)

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)
response.output_text

## 7. Automated Agentic Loop

Now let's automate the tool-calling cycle with a helper function and a loop.

In [ ]:
def make_call(call):
    """Execute a tool call and return the result in the expected format."""
    args = json.loads(call.arguments)
    f_name = call.name
    f = globals()[f_name]
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

If you want to look up the answer, explain why before making the call. Use as many 
keywords from the user question as possible when making first requests.

Make multiple searches. Try to expand your search by using new keywords based on the results you
get from the search.

At the end, make a clarifying question based on what you presented and ask if there are 
other areas that the user wants to explore.
""".strip()

In [ ]:
question = "I just discovered the course, can I join it now?"

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

In [ ]:
# Automated agentic loop
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=chat_messages,
        tools=[search_tool]
    )
    
    chat_messages.extend(response.output)

    has_function_calls = False
    
    for entry in response.output:
        if entry.type == 'message':
            print(entry.content[0].text)
        if entry.type == 'function_call':
            print(f"[Tool call: {entry.name}({entry.arguments})]")
            result = make_call(entry)
            chat_messages.append(result)
            has_function_calls = True

    if not has_function_calls:
        break